# Práctica 1: Predicciones de Competición
**Predicción de Subscripción a un Producto Bancario**

* **Autores:** Jaime del Campo Montaña & David Alexander Torres Paillacho
* **NIA's:** 100495957 & 100522187

---
Este notebook **carga el modelo final ya entrenado** (`modelo_final.joblib`) y lo aplica sobre el conjunto de competición (`bank_competition_15.pkl`) para generar el fichero `predicciones.csv`.



Aqui unicamente se uso IA Generativa para corregir errores menores, se le paso el notebook celda por celda y fue ajustando detalles. 
Ademas se uso IA para comprobar que todo funcionase correctamente.


In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import FunctionTransformer

#necesario para que joblib pueda reconstruir el pipeline guardado

def transform_pdays(X):
    vals = np.array(X).flatten()
    no_previo = (vals == -1).astype(int)
    dias = np.where(vals == -1, 0, vals)
    return np.column_stack((no_previo, dias))

def get_pdays_names(transformer, feature_names):
    return ['pdays_no_previo', 'pdays_dias']

print("Librerías y funciones auxiliares cargadas.")

✅ Librerías y funciones auxiliares cargadas.


## 1. Carga del modelo final

In [ ]:
modelo = joblib.load('modelo_final.joblib')
print(f"Modelo cargado: {modelo.steps[-1][0]}")
print(f"   Pasos del pipeline: {[nombre for nombre, _ in modelo.steps]}")

✅ Modelo cargado: logreg
   Pasos del pipeline: ['preprocessor', 'scaler', 'logreg']


## 2. Carga del conjunto de competición

In [ ]:
df_comp = pd.read_pickle('bank_competition.pkl')

print(f"Dataset de competición cargado: {df_comp.shape[0]} filas × {df_comp.shape[1]} columnas")
print(f"Columnas: {list(df_comp.columns)}")
df_comp.head()

✅ Dataset de competición cargado: 162 filas × 16 columnas
   Columnas: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
5553,43,management,married,tertiary,no,78,yes,no,cellular,21,nov,36,1,109,1,other
915,34,housemaid,married,secondary,no,0,yes,no,unknown,30,oct,154,1,-1,0,unknown
7652,54,technician,married,secondary,no,3323,yes,yes,cellular,8,apr,59,3,-1,0,unknown
5065,43,blue-collar,single,primary,no,-399,no,yes,cellular,28,jul,662,3,-1,0,unknown
3338,35,blue-collar,married,secondary,no,262,no,no,cellular,15,mar,427,1,181,3,success


## 3. Generación de predicciones

El pipeline incluye internamente todo el preprocesamiento (incluyendo la transformación de `pdays`). **No es necesario hacer ningún preproceso manual.**

In [ ]:
predicciones_num = modelo.predict(df_comp)

#convertimos de vuelta a etiquetas 'yes'/'no'
predicciones_label = pd.Series(predicciones_num).map({1: 'yes', 0: 'no'})

print(f"Predicciones generadas: {len(predicciones_num)} instancias")
print(f"Distribución: {predicciones_label.value_counts().to_dict()}")

✅ Predicciones generadas: 162 instancias
   Distribución: {'no': 103, 'yes': 59}


## 4. Verificación — Comparación de dos instancias

Comprobamos manualmente las predicciones para dos instancias concretas. Estos mismos valores deben coincidir con los que muestre la app de Streamlit.

In [6]:
instancias_verificacion = df_comp.iloc[[0, 1]]

preds_verificacion = modelo.predict(instancias_verificacion)
preds_labels_verif = pd.Series(preds_verificacion).map({1: 'yes', 0: 'no'})

print("── Verificación instancias 0 y 1 ──")
print(instancias_verificacion.to_string())
print(f"\nPredicción instancia 0: {preds_labels_verif.iloc[0]}")
print(f"Predicción instancia 1: {preds_labels_verif.iloc[1]}")
print("\n→ Anota estos valores. Deben coincidir exactamente con los de la app Streamlit.")

── Verificación instancias 0 y 1 ──
      age         job  marital  education default  balance housing loan   contact  day month  duration  campaign  pdays  previous poutcome
5553   43  management  married   tertiary      no       78     yes   no  cellular   21   nov        36         1    109         1    other
915    34   housemaid  married  secondary      no        0     yes   no   unknown   30   oct       154         1     -1         0  unknown

Predicción instancia 0: no
Predicción instancia 1: no

→ Anota estos valores. Deben coincidir exactamente con los de la app Streamlit.


## 5. Guardado de predicciones

In [ ]:
df_output = pd.DataFrame({'deposit': predicciones_label.values})
df_output.to_csv('predicciones.csv', index=False)

print("Predicciones guardadas en 'predicciones.csv'")
print(f"Total filas: {len(df_output)}")
df_output.head(10)

✅ Predicciones guardadas en 'predicciones.csv'
   Total filas: 162


,deposit
0,no
1,no
2,no
3,yes
4,yes
5,no
6,yes
7,no
8,yes
9,no
